# Week 23, Day 1: Regular expressions I

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w023d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import re

text = "Call 0161 496 0000 or 020 7946 0018 before 5pm."
print(re.findall(r"\d+", text))
print(re.findall(r"\d{4}", text))

In [ ]:
import re

m = re.search(r"\d+", "Room 101, floor 3")
if m:
    print(m.group(), m.span())

print(re.search(r"\d+", "no numbers here"))
print(bool(re.fullmatch(r"\d+", "2026")), bool(re.fullmatch(r"\d+", "2026a")))

for m in re.finditer(r"\d+", "Room 101, floor 3"):
    print(m.group(), "at", m.start())

*Interactive figure on the website: Edit the pattern or the text, or pick a pattern below. Matches are highlighted in turn..*

In [ ]:
import re

print(re.findall(r"[aeiou]", "regular expression"))
print(re.findall(r"[^aeiou ]", "regular"))
print(re.findall(r"b.t", "bat bet bit b?t bt"))
print(re.findall(r"\w", "a_1 !"), re.findall(r"\s", "a b\tc"))

In [ ]:
import re

print(re.findall(r"colou?r", "color colour colouur"))
print(re.findall(r"\d{3,}", "7 42 365 2026"))
print(re.fullmatch(r"\d{4}", "0420") is not None, re.fullmatch(r"\d{4}", "042") is not None)
print(re.findall(r"[A-Z][a-z]+", "Ada met Bisi and Chen in Leeds"))

In [ ]:
import re

text = "cat category bobcat cat."
print(re.findall(r"cat", text))
print(re.findall(r"\bcat\b", text))
print(re.findall(r"\b(?:Mon|Tues|Wednes)day\b", "Monday, Tuesday and Sunday"))

lines = "apple 3\nbanana 12\ncherry 7"
print(re.findall(r"^\w+", lines))
print(re.findall(r"^\w+", lines, re.MULTILINE))

In [ ]:
import re

print(re.findall("\bcat\b", "a cat"))        # "\b" became a backspace: no match
print(re.findall(r"\bcat\b", "a cat"))       # raw string: works

In [ ]:
import re

print(re.findall(r"\$\d+\.\d\d", "Total $14.99, tax $1.20"))
price = "$14.99"
print(re.escape(price), re.search(re.escape(price), "Total $14.99") is not None)

> **Key idea.** `re.search` finds the first match, `re.fullmatch` checks the whole text, `re.findall` lists every match. Build patterns from characters, `.`, classes (`[a-z]`, `\d`, `\w`, `\s`), quantifiers (`*`, `+`, `?`, `{m,n}`), anchors (`^`, `$`, `\b`) and `|`. Write patterns as raw strings, `r"..."`, and escape special characters with `\` or `re.escape`.

### Exercise: Find them

Using `re.findall`, set each variable.

In [ ]:
import re

TEXT = "Ada (32) paid £18.50; Ben (27) paid £7.25 and Cy (41) paid £120.00 on 31/10/2026."

ages = ___      # the three ages with their brackets: ["(32)", "(27)", "(41)"]
names = ___     # the capitalised names: ["Ada", "Ben", "Cy"]
amounts = ___   # the amounts as strings without £: ["18.50", "7.25", "120.00"]
date = ___      # the date as a string, using search(...).group()

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert ages == [\"(32)\", \"(27)\", \"(41)\"] and names == [\"Ada\", \"Ben\", \"Cy\"]\n    assert amounts == [\"18.50\", \"7.25\", \"120.00\"] and date == \"31/10/2026\"\n    assert \"split\" not in source")

### Exercise: Validate input

Write each check with `re.fullmatch`, returning `True` or `False`:

- `is_pin(s)`: exactly 4 digits.
- `is_username(s)`: 3 to 16 characters, letters, digits or underscores, starting with a letter.
- `is_hex_colour(s)`: `#` followed by exactly 6 hexadecimal digits (0-9, a-f, either case).
- `is_time(s)`: a 24-hour time `HH:MM` from `00:00` to `23:59`.

In [ ]:
import re

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert is_pin(\"0420\") and not is_pin(\"042\") and not is_pin(\"04200\") and not is_pin(\"04a0\")\n    assert is_username(\"ada_99\") and is_username(\"Bob\") and not is_username(\"9lives\") and not is_username(\"ab\") and not is_username(\"x\" * 17)\n    assert is_hex_colour(\"#1F2a3c\") and not is_hex_colour(\"1F2a3c\") and not is_hex_colour(\"#12345g\")\n    assert is_time(\"00:00\") and is_time(\"23:59\") and is_time(\"09:05\") and not is_time(\"24:00\") and not is_time(\"9:05\") and not is_time(\"12:60\")")

### Exercise: Whole words

Write `count_word(text, word)` returning how many times `word` appears as a **whole word**, ignoring case. The word may contain characters that are special in regex (like `C++` or `3.5`), so escape it.

In [ ]:
import re

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    t = \"The cat sat. Concatenate CAT, catalogue, cat!\"\n    assert count_word(t, \"cat\") == 3\n    assert count_word(\"I like C and C++, not C#.\", \"C++\") == 1\n    assert count_word(\"version 3.5 and 345\", \"3.5\") == 1")

### Exercise: Line by line

A log has lines like `2026-10-31 14:02:11 ERROR Disk full`. Write `levels(log)` returning a list of the level word (`INFO`, `WARNING`, `ERROR` or `DEBUG`) from each line that has one, in order, using one `re.findall` with `re.MULTILINE` and a `^` anchor; and `errors_at(log)` returning the times (`HH:MM:SS`) of the `ERROR` lines. One preview of tomorrow: if a pattern contains one part in round brackets, `findall` returns just that part of each match, so `re.findall(r"^\S+ \S+ (INFO|ERROR)", log, re.M)` gives the level words.

In [ ]:
import re

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    log = (\"2026-10-31 14:02:11 INFO Started\\n\"\n           \"2026-10-31 14:02:15 ERROR Disk full\\n\"\n           \"    continued text with ERROR inside\\n\"\n           \"2026-10-31 14:03:01 WARNING Slow\\n\"\n           \"2026-10-31 14:05:42 ERROR Timeout\\n\")\n    assert levels(log) == [\"INFO\", \"ERROR\", \"WARNING\", \"ERROR\"]\n    assert errors_at(log) == [\"14:02:15\", \"14:05:42\"]")

### Exercise: UK postcodes

UK postcodes look like `LS1 4AB`, `M1 1AE`, `SW1A 2AA` or `B33 8TH`: one or two letters, a digit, then optionally a letter or digit, a space, then a digit and two letters. Write `postcodes(text)` returning all the postcodes in a text (capital letters only), using a `\b` at each end.

In [ ]:
import re

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    t = \"Send to LS1 4AB or SW1A 2AA. Not ls1 4ab, not LS1-4AB, and B33 8TH too. M1 1AE!\"\n    assert postcodes(t) == [\"LS1 4AB\", \"SW1A 2AA\", \"B33 8TH\", \"M1 1AE\"]\n    assert postcodes(\"none here\") == []")

### Bug hunt: A dot is not a dot

This should find prices like `3.50`, but it also matches `3x50` and `3 50`, because `.` means "any character". Fix the pattern so it prints `['3.50', '12.00']`.

In [ ]:
import re
print(re.findall(r"\d+.\d\d", "3.50 or 3x50 or 3 50 or 12.00"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"['3.50', '12.00']\")")

### Bug hunt: search is not fullmatch

This PIN check accepts `"12345"` and `"PIN 1234!"`, because `re.search` finds four digits anywhere. Fix it so it prints `[True, False, False, False]`.

In [ ]:
import re

def is_pin(s):
    return re.search(r"\d{4}", s) is not None

print([is_pin(s) for s in ["1234", "12345", "PIN 1234!", "12a4"]])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"[True, False, False, False]\")")

### Puzzle: Find hashtags

Arrange the lines so the program prints the hashtags in a post, without the `#`, in lower case. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them.

In [ ]:
import re

a = ___   # re.findall(r"a.c", "abc a-c ac a\nc")
b = ___   # re.findall(r"[^aeiou]+", "beautiful")
c = ___   # bool(re.fullmatch(r"ha+", "haaa"))
d = ___   # re.findall(r"\bon\b", "on one only on")  (how many? a number)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == [\"abc\", \"a-c\"] and b == [\"b\", \"t\", \"f\", \"l\"] and c is True and d == 2")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: tidy up phone numbers

People type UK phone numbers in many ways: `0113 496 0000`, `(0113) 4960000`, `+44 113 496 0000`, `0044-113-496-0000`. Write:

- `find_numbers(text)`: every phone number in a text, as written, using one regex: an optional `+44` or `0044` (then optional spaces or hyphens) **or** a `0`, optionally inside brackets with the area code; followed by digits, spaces and hyphens, such that there are 10 or 11 digits in total. (Find candidates with a generous pattern, then keep those whose digit count, once `+44`/`0044` is turned into `0`, is exactly 11.)
- `normalise(number)`: the number as 11 digits starting with `0`, with no spaces or punctuation (`+44 113 496 0000` becomes `01134960000`), or `None` if it isn't valid.

In [ ]:
import re

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    t = \"Call 0113 496 0000 or (0113) 4960000, abroad +44 113 496 0000 or 0044-113-496-0000. Not 12345 or 0113 49.\"\n    found = find_numbers(t)\n    assert [normalise(n) for n in found] == [\"01134960000\"] * 4\n    assert normalise(\"07700 900123\") == \"07700900123\" and normalise(\"12345\") is None and normalise(\"+44 7700 900 123\") == \"07700900123\"")

## Warm-up

`\d` means "a digit", `+` means "one or more", and `{4}` means "exactly four". Two tiny patterns, two very different answers, and no loops. That's the appeal of regular expressions.

## Searching

Python's `re` module has a few ways to search:

| Function | Looks for | Returns |
|---|---|---|
| `re.search(p, text)` | the **first** match anywhere | a match object, or `None` |
| `re.match(p, text)` | a match at the **start** only | a match object, or `None` |
| `re.fullmatch(p, text)` | the **whole** text matching | a match object, or `None` |
| `re.findall(p, text)` | every match | a list of strings |
| `re.finditer(p, text)` | every match | match objects, one at a time |

A **match object** tells you what matched and where: `m.group()` (or `m.group(0)`) is the matched text, `m.start()` and `m.end()` its position, `m.span()` both. Because `search` gives `None` when nothing matches, check before using it:

`fullmatch` is the one for **checking** input ("is this a valid PIN?"); `search` and `findall` are for **finding** things inside text.

Try patterns here. Everything runs with Python's own `re`:

## Characters and classes

Most characters in a pattern simply match themselves: `cat` matches "cat". A few have special meanings:

| Pattern | Matches |
|---|---|
| `.` | any one character except a newline |
| `[abc]` | one character: `a`, `b` or `c` |
| `[a-z]`, `[A-Z0-9]` | one character in a range (or several ranges) |
| `[^0-9]` | one character that's **not** in the set (a `^` at the start) |
| `\d` | a digit, same as `[0-9]` (for plain digits) |
| `\w` | a "word" character: letter, digit or underscore |
| `\s` | whitespace: space, tab, newline |
| `\D`, `\W`, `\S` | the opposite: not a digit, not a word character, not whitespace |

Inside `[...]`, most special characters lose their meaning: `[.?]` matches a full stop or a question mark. A `-` means a range unless it comes first or last: `[-+]` is a minus or a plus.

## Repeats

A **quantifier** after something says how many times it repeats:

| Quantifier | Means |
|---|---|
| `*` | 0 or more times |
| `+` | 1 or more times |
| `?` | 0 or 1 time (optional) |
| `{3}` | exactly 3 times |
| `{2,4}` | 2 to 4 times |
| `{2,}` | 2 or more times |

It applies to the single thing just before it: `ab+` is "a, then one or more b"; to repeat more than one character, group it with brackets (tomorrow).

Quantifiers are **greedy**: they match as much as they can. `<.+>` on `"<b>bold</b>"` matches the whole string, not just `<b>`. Tomorrow you'll see how to make them match as little as possible.

## Anchors and alternatives

**Anchors** match a **position**, not a character:

| Anchor | Matches at |
|---|---|
| `^` | the start of the text (or of each line, with `re.MULTILINE`) |
| `$` | the end of the text (or of each line, with `re.MULTILINE`) |
| `\b` | a word boundary: between a word character and a non-word character |

`\b` is surprisingly useful: `\bcat\b` finds "cat" but not "category" or "concatenate".

**Alternatives:** `|` means "or": `cat|dog` matches either. It has the lowest priority, so `^cat|dog$` means "cat at the start, or dog at the end".

`(?:...)` groups alternatives without capturing them; you'll see why that matters tomorrow.

**Flags** change how a pattern behaves: `re.IGNORECASE` (or `re.I`) ignores capitals, `re.MULTILINE` (`re.M`) makes `^` and `$` work per line. Combine them with `|`: `re.I | re.M`.

## Raw strings and escaping

Backslashes mean something to Python strings **and** to regex: in an ordinary string, `"\b"` is a backspace character, not a word boundary. Always write patterns as **raw strings**, with an `r` before the quote, so backslashes reach `re` untouched:

To match a special character **literally**, put a backslash before it: `\.` is a real full stop, `\$` a dollar sign, `\(` a bracket. `re.escape(text)` does it for you, which is the safe way to search for text a user typed:

## Practice

## Quiz

## Daily challenge

## Before you go

Open the regex lab above and write a pattern for something in your own life: your country's phone numbers, your student ID format, or dates the way you write them. Tomorrow: groups (pulling out parts of a match), substitution, and splitting with patterns.